# spaCy

## Definition
spaCy is a fast, industrial-strength Python library for advanced NLP. It provides pre-trained statistical models for tokenization, POS tagging, dependency parsing, Named Entity Recognition (NER), and text classification. spaCy is designed for production use and is significantly faster than NLTK due to its Cython-optimized core.

## Why It Is Needed
- **Production-Ready Speed:** Processes large volumes of text faster than any other Python NLP library using Cython internals.
- **Pipeline Architecture:** Offers a modular, extendable pipeline where custom components can be added, replaced, or disabled with ease.
- **Pre-trained Models:** Provides ready-to-use statistical models for 60+ languages, removing the need to train from scratch.

## Real-World Applications
- Named Entity Recognition in information extraction systems
- Dependency parsing for knowledge graph construction
- Text preprocessing pipelines in production NLP applications
- Building custom NER models for domain-specific text (medical, legal)
- Entity linking and coreference resolution in document analysis

## Important Points
- **Pipeline Components:** `tokenizer → tagger → parser → ner → lemmatizer → textcat`
- **Language Models:**
  - `en_core_web_sm` — Small, fast (12 MB)
  - `en_core_web_md` — Medium, includes word vectors (43 MB)
  - `en_core_web_lg` — Large, best accuracy (741 MB)
  - `en_core_web_trf` — Transformer-based, most accurate
- **`Doc`, `Token`, `Span`:** Core spaCy objects — `Doc` is the processed text container, `Token` represents a single word, `Span` is a sequence of tokens.
- **`nlp.pipe()`:** Processes multiple texts in batches for maximum throughput.
- **displaCy:** spaCy's built-in visualizer for dependency trees and named entities.



## Implementation
Practical implementation will be added here.

## Key Takeaways
- spaCy is the industry-standard Python library for production NLP pipelines.
- It uses a modular pipeline architecture where components can be added or disabled.
- `Doc`, `Token`, and `Span` are the three core data structures in spaCy.
- `nlp.pipe()` enables batch processing for high-throughput text analysis.
- spaCy supports transformer-based models via the `spacy-transformers` extension.

NER Visualization with Displacy for Legal Contracts

In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [2]:
from spacy import displacy

doc = nlp("This Agreement is made on 4th April 2023 between Apple Inc. and John Doe.")
displacy.render(doc, style="ent", jupyter=True)


Custom Pattern Matching (e.g., Date + Entity)

In [3]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
pattern = [{"ENT_TYPE": "DATE"}, {"LOWER": "between"}, {"ENT_TYPE": "ORG"}]
matcher.add("DATE_CONTRACT_PATTERN", [pattern])

doc = nlp("Signed on 5th March 2024 between Microsoft and John.")
matches = matcher(doc)

for match_id, start, end in matches:
    print(doc[start:end].text)

2024 between Microsoft


Text Clustering using spaCy Vectors

In [4]:
from sklearn.cluster import KMeans
import numpy as np

sentences = ["The cat sits on the mat", "Dogs bark loudly", "The dog is in the yard"]
docs = [nlp(sent) for sent in sentences]
X = np.array([doc.vector for doc in docs])

kmeans = KMeans(n_clusters=2)
kmeans.fit(X)

for i, label in enumerate(kmeans.labels_):
    print(f"Cluster {label}: {sentences[i]}")

Cluster 0: The cat sits on the mat
Cluster 1: Dogs bark loudly
Cluster 0: The dog is in the yard


Fine-Grained NER Training (Custom Labels)

In [5]:
import spacy
from spacy.training.example import Example
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# Start from a blank English model
nlp = spacy.blank("en")

# Add NER pipeline component
ner = nlp.add_pipe("ner")
ner.add_label("ORG")
ner.add_label("PRODUCT")

# Create a sample training set with diverse examples
TRAIN_DATA = [
    ("Apple released the new Vision Pro.", {"entities": [(0, 5, "ORG"), (24, 34, "PRODUCT")]}),
    ("Microsoft launched Surface Laptop.", {"entities": [(0, 9, "ORG"), (19, 34, "PRODUCT")]}),
    ("Google unveiled the Pixel 8 phone.", {"entities": [(0, 6, "ORG"), (19, 29, "PRODUCT")]}),
    ("Apple Vision Pro is a mixed reality headset.", {"entities": [(0, 5, "ORG"), (6, 16, "PRODUCT")]}),
    ("Amazon presented the new Echo Show 10.", {"entities": [(0, 6, "ORG"), (25, 39, "PRODUCT")]}),
]

# Convert to Example objects
examples = []
for text, annots in TRAIN_DATA:
    doc = nlp.make_doc(text)
    examples.append(Example.from_dict(doc, annots))

# Training loop
optimizer = nlp.begin_training()
for i in range(30):  # More iterations for better training
    nlp.update(examples, sgd=optimizer)

# Test
test_doc = nlp("Apple introduced the Vision Pro headset today.")
print([(ent.text, ent.label_) for ent in test_doc.ents])

[('Apple', 'ORG'), ('Vision Pro', 'PRODUCT')]
